# Practice: L1 (Lasso), L2 (Ridge), and Elastic Net Regularizations
Synthetic Gene Expression Dataset (`300 samples x 500 genes`, only `15` informative genes).

### 1. Importing Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

### 2. Generate Synthetic Gene Expression Data

In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=500,
    n_informative=15,
    n_redundant=20,
    n_repeated=0,
    n_classes=2,
    random_state=42
)

gene_names = [f"Gene_{i+1}" for i in range(500)]
df = pd.DataFrame(X, columns=gene_names)
df["Class"] = y
print("Dataset shape:", df.shape)
display(df.head())

### 3. Split Dataset (80/20 Stratified)

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

### 4. Train 3 Different Models (L1, L2, Elastic Net) with Feature Scaling

In [ ]:
models = {
    "L1 (Lasso)": LogisticRegression(
        penalty="l1",
        solver="saga",
        C=0.1,
        max_iter=10000,
        random_state=42
    ),
    "L2 (Ridge)": LogisticRegression(
        penalty="l2",
        solver="saga",
        C=0.1,
        max_iter=10000,
        random_state=42
    ),
    "Elastic Net": LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=0.1,
        l1_ratio=0.5,
        max_iter=10000,
        random_state=42
    )
}

trained_models = {}
for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])
    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline
    print(name, "training completed")

### 5. Compare Results

In [ ]:
results = []
for name, model in trained_models.items():
    predictions = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1 Score": f1_score(y_test, predictions, zero_division=0)
    })
results_df = pd.DataFrame(results)
display(results_df.round(3))

### 6. Visualise Feature Selection & Number of Retained Features

In [ ]:
coefficient_data = {}
feature_counts = {}
for name, model in trained_models.items():
    coefficients = model.named_steps["classifier"].coef_[0]
    coefficient_data[name] = coefficients
    nonzero = np.sum(np.abs(coefficients) > 1e-6)
    feature_counts[name] = nonzero
    print(name, "Non-zero features:", nonzero, "/ 500")

plt.figure(figsize=(9, 5))
plt.bar(feature_counts.keys(), feature_counts.values(), color=["#2563eb", "#059669", "#7c3aed"])
plt.ylabel("Number of non-zero coefficients")
plt.title("Feature retention: L1 vs L2 vs Elastic Net")
plt.show()

### 7. Demonstrate Effect of Regularization Strength (C Values on L1)

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
nonzero_counts = []
for C_value in C_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            penalty="l1",
            solver="saga",
            C=C_value,
            max_iter=10000,
            random_state=42
        ))
    ])
    model.fit(X_train, y_train)
    coefficients = model.named_steps["classifier"].coef_[0]
    nonzero_counts.append(np.sum(np.abs(coefficients) > 1e-6))

plt.figure(figsize=(9, 5))
plt.plot(C_values, nonzero_counts, marker="o", linewidth=2, color="#dc2626")
plt.xscale("log")
plt.xlabel("C (inverse regularization strength)")
plt.ylabel("Number of non-zero coefficients")
plt.title("Effect of regularization strength on L1")
plt.grid(True)
plt.show()